In [ ]:
# ===============================================================
# NEX-GDDP-CMIP6  |  Monthly cumulative precipitation (mm/month)
# AOI: Abay Basin  |  Author: ChatGPT (GPT-5)
# ===============================================================

import ee
ee.Authenticate()      # run once interactively
ee.Initialize()

# ---------------------------------------------------------------
# SETTINGS
# ---------------------------------------------------------------
AOI = ee.FeatureCollection("projects/lithe-inscriber-448612-q7/assets/abay")
MODEL = 'ACCESS-CM2'
SCALE = 25000
FOLDER = 'NEXGDDP_CMIP6_Abay_Monthly'

SCENARIOS = ['historical', 'ssp126', 'ssp245', 'ssp370', 'ssp585']
PERIODS = {
    'historical': {'start': 1981, 'end': 2015},
    'mid': {'start': 2041, 'end': 2070},
    'far': {'start': 2071, 'end': 2100}
}

# ---------------------------------------------------------------
# HELPERS
# ---------------------------------------------------------------

def convert_to_mm_per_day_and_clip(img):
    """Convert 'pr' from kg m-2 s-1 → mm/day and clip to AOI."""
    return (img.select('pr')
            .multiply(86400)
            .clip(AOI)
            .copyProperties(img, img.propertyNames()))

def monthly_sum_for_year(model, scenario, year):
    """Return an ImageCollection of 12 monthly-sum images for a given year."""
    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, 'year')

    daily = (ee.ImageCollection('NASA/GDDP-CMIP6')
             .filter(ee.Filter.eq('model', model))
             .filter(ee.Filter.eq('scenario', scenario))
             .filterDate(start, end)
             .map(convert_to_mm_per_day_and_clip))

    months = ee.List.sequence(1, 12)

    def make_month(m):
        m = ee.Number(m)
        m_start = ee.Date.fromYMD(year, m, 1)
        m_end = m_start.advance(1, 'month')
        monthly = (daily.filterDate(m_start, m_end)
                   .sum()
                   .set({'year': year,
                         'month': m,
                         'scenario': scenario,
                         'model': model,
                         'system:time_start': m_start.millis()}))
        return monthly

    return ee.ImageCollection(months.map(make_month))

def export_monthly_collection(coll, prefix, folder, region, scale):
    """Create one Drive export per image in a small ImageCollection."""
    size = coll.size().getInfo()
    lst = coll.toList(size)
    for i in range(size):
        img = ee.Image(lst.get(i))
        year = img.get('year').getInfo()
        month = img.get('month').getInfo()
        mm = f'{int(month):02d}'
        desc = f'{prefix}_{year}_{mm}'
        task = ee.batch.Export.image.toDrive(
            image=img,
            description=desc,
            folder=folder,
            fileNamePrefix=desc,
            region=region,
            scale=scale,
            maxPixels=1e13
        )
        task.start()
        print(f'Started export: {desc}')

# ---------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------

for scenario in SCENARIOS:
    if scenario == 'historical':
        # --- Historical (1981-2015)
        period = PERIODS['historical']
        print(f'Processing historical: {period["start"]}-{period["end"]}')
        for year in range(period['start'], period['end'] + 1):
            monthly = monthly_sum_for_year(MODEL, scenario, year)
            prefix = f'{MODEL}_{scenario}'
            export_monthly_collection(monthly, prefix, FOLDER,
                                      AOI.geometry(), SCALE)
    else:
        # --- Mid-future (2041-2070)
        mid = PERIODS['mid']
        print(f'Processing {scenario} mid: {mid["start"]}-{mid["end"]}')
        for year in range(mid['start'], mid['end'] + 1):
            monthly = monthly_sum_for_year(MODEL, scenario, year)
            prefix = f'{MODEL}_{scenario}_mid'
            export_monthly_collection(monthly, prefix, FOLDER,
                                      AOI.geometry(), SCALE)

        # --- Far-future (2071-2100)
        far = PERIODS['far']
        print(f'Processing {scenario} far: {far["start"]}-{far["end"]}')
        for year in range(far['start'], far['end'] + 1):
            monthly = monthly_sum_for_year(MODEL, scenario, year)
            prefix = f'{MODEL}_{scenario}_far'
            export_monthly_collection(monthly, prefix, FOLDER,
                                      AOI.geometry(), SCALE)

print("✅ All export tasks submitted. Check https://code.earthengine.google.com/tasks")


### Final Python Script — Historical Only (1981–2015, EC-Earth3)

In [3]:
# ===============================================================
# NEX-GDDP-CMIP6  |  Monthly cumulative precipitation (mm/month)
# AOI: Abay Basin  |  Model: NorESM2-LM | Period: 1981–2015
# ===============================================================

import ee
ee.Authenticate()      # run once interactively
ee.Initialize()

# ---------------------------------------------------------------
# SETTINGS
# ---------------------------------------------------------------
AOI = ee.FeatureCollection("projects/steam-treat-476613-h0/assets/abay")
MODEL = 'NorESM2-LM'
SCALE = 25000
FOLDER = 'NEX-GDDP-CMIP6_Abbay_Monthly_NorESM2-LM'

SCENARIO = 'historical'
PERIOD = {'start': 1981, 'end': 2015}

# ---------------------------------------------------------------
# HELPERS
# ---------------------------------------------------------------

def convert_to_mm_per_day_and_clip(img):
    """Convert 'pr' from kg m-2 s-1 → mm/day and clip to AOI."""
    return (img.select('pr')
            .multiply(86400)
            .clip(AOI.union())
            .copyProperties(img, img.propertyNames()))

def monthly_sum_for_year(model, scenario, year):
    """Return an ImageCollection of 12 monthly-sum images for a given year."""
    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, 'year')

    daily = (ee.ImageCollection('NASA/GDDP-CMIP6')
             .filter(ee.Filter.eq('model', model))
             .filter(ee.Filter.eq('scenario', scenario))
             .filterDate(start, end)
             .map(convert_to_mm_per_day_and_clip))

    months = ee.List.sequence(1, 12)

    def make_month(m):
        m = ee.Number(m)
        m_start = ee.Date.fromYMD(year, m, 1)
        m_end = m_start.advance(1, 'month')
        monthly = (daily.filterDate(m_start, m_end)
                   .sum()
                   .set({'year': year,
                         'month': m,
                         'scenario': scenario,
                         'model': model,
                         'system:time_start': m_start.millis()}))
        return monthly

    return ee.ImageCollection(months.map(make_month))

def export_monthly_collection(coll, prefix, folder, region, scale):
    """Create one Drive export per image in a small ImageCollection."""
    region = AOI.union().geometry()
    size = coll.size().getInfo()
    lst = coll.toList(size)
    for i in range(size):
        img = ee.Image(lst.get(i))
        year = img.get('year').getInfo()
        month = img.get('month').getInfo()
        mm = f'{int(month):02d}'
        desc = f'{prefix}_{year}_{mm}'
        task = ee.batch.Export.image.toDrive(
            image=img,
            description=desc,
            folder=folder,
            fileNamePrefix=desc,
            region=region,
            scale=scale,
            maxPixels=1e13
        )
        task.start()
        print(f'Started export: {desc}')

# ---------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------

print(f'Processing {MODEL} historical: {PERIOD["start"]}-{PERIOD["end"]}')
for year in range(PERIOD['start'], PERIOD['end'] + 1):
    monthly = monthly_sum_for_year(MODEL, SCENARIO, year)
    prefix = f'{MODEL}_{SCENARIO}'
    export_monthly_collection(monthly, prefix, FOLDER, AOI.geometry(), SCALE)

print("✅ All export tasks submitted. Check https://code.earthengine.google.com/tasks")


Processing NorESM2-LM historical: 1981-2015
Started export: NorESM2-LM_historical_1981_01
Started export: NorESM2-LM_historical_1981_02
Started export: NorESM2-LM_historical_1981_03
Started export: NorESM2-LM_historical_1981_04
Started export: NorESM2-LM_historical_1981_05
Started export: NorESM2-LM_historical_1981_06
Started export: NorESM2-LM_historical_1981_07
Started export: NorESM2-LM_historical_1981_08
Started export: NorESM2-LM_historical_1981_09
Started export: NorESM2-LM_historical_1981_10
Started export: NorESM2-LM_historical_1981_11
Started export: NorESM2-LM_historical_1981_12
Started export: NorESM2-LM_historical_1982_01
Started export: NorESM2-LM_historical_1982_02
Started export: NorESM2-LM_historical_1982_03
Started export: NorESM2-LM_historical_1982_04
Started export: NorESM2-LM_historical_1982_05
Started export: NorESM2-LM_historical_1982_06
Started export: NorESM2-LM_historical_1982_07
Started export: NorESM2-LM_historical_1982_08
Started export: NorESM2-LM_historica

In [1]:
# 1. Install the Earth Engine API and geemap (for easier visualization)
!pip install earthengine-api geemap

# 2. Import the libraries
import ee
import geemap

# 3. Authenticate and Initialize
# This will open a window to log into your Google Earth Engine account
try:
    ee.Initialize()
    print("Google Earth Engine initialized successfully!")
except Exception as e:
    print("Authentication required...")
    ee.Authenticate()
    ee.Initialize()

# 4. Quick Test: Print EC-Earth3 Metadata
dataset = ee.ImageCollection('NASA/GDDP-CMIP6') \
            .filter(ee.Filter.eq('model', 'EC-Earth3')) \
            .limit(1)

print("Successfully connected. Sample Metadata:", dataset.first().getInfo())

Defaulting to user installation because normal site-packages is not writeable


Matplotlib is building the font cache; this may take a moment.


Authentication required...


Enter verification code:  4/1Aci98E_CzEplz4ODQ9YLe3xdhaOPlDG9yRyasHCMURwzMbhOptIvGWVlZNA



Successfully saved authorization token.
Successfully connected. Sample Metadata: {'type': 'Image', 'bands': [{'id': 'hurs', 'data_type': {'type': 'PixelType', 'precision': 'float'}, 'dimensions': [1441, 600], 'crs': 'EPSG:4326', 'crs_transform': [0.25, 0, -180, 0, -0.25, 90]}, {'id': 'huss', 'data_type': {'type': 'PixelType', 'precision': 'float'}, 'dimensions': [1441, 600], 'crs': 'EPSG:4326', 'crs_transform': [0.25, 0, -180, 0, -0.25, 90]}, {'id': 'pr', 'data_type': {'type': 'PixelType', 'precision': 'float'}, 'dimensions': [1441, 600], 'crs': 'EPSG:4326', 'crs_transform': [0.25, 0, -180, 0, -0.25, 90]}, {'id': 'rlds', 'data_type': {'type': 'PixelType', 'precision': 'float'}, 'dimensions': [1441, 600], 'crs': 'EPSG:4326', 'crs_transform': [0.25, 0, -180, 0, -0.25, 90]}, {'id': 'rsds', 'data_type': {'type': 'PixelType', 'precision': 'float'}, 'dimensions': [1441, 600], 'crs': 'EPSG:4326', 'crs_transform': [0.25, 0, -180, 0, -0.25, 90]}, {'id': 'sfcWind', 'data_type': {'type': 'PixelT